# LightGBM

Gradient boosting framework using leaf-wise (best-first) tree growth strategy. Faster than traditional level-wise approaches on large datasets, with native support for categorical features and efficient handling of sparse data.

**When to Use:**
- Large datasets where training speed matters
- High-cardinality categorical features (native categorical support)
- Tabular data with complex non-linear patterns
- When you need a fast, accurate gradient boosting model

**Key Assumptions / Considerations:**
- Leaf-wise growth can overfit on small datasets — control with num_leaves
- num_leaves is more important than max_depth (leaf-wise vs level-wise)
- Native categorical handling avoids one-hot encoding explosion
- Feature fraction (colsample_bytree) and bagging help prevent overfitting
- Early stopping recommended for optimal number of boosting rounds

**References:**
- [LightGBM Documentation](https://lightgbm.readthedocs.io/en/stable/)
- [LightGBM Python API](https://lightgbm.readthedocs.io/en/stable/Python-API.html)
- [LightGBM Parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html)

In [ ]:
# pip install lightgbm

import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
)

from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)
target_binary = (target_continuous > np.median(target_continuous)).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_continuous  # switch to target_binary for classification 
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Distribution

plt.figure(figsize=(8, 5))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

print("Skewness:", round(y.skew(),4))
print("Kurtosis:", round(y.kurt(), 4))

In [ ]:
# Train/Test Split

try:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    # if y is continuous
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "LGBM_Regressor": (LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [-1, 5, 10],
        "clf__num_leaves": [31, 63, 127],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__reg_alpha": [0, 0.1],
        "clf__reg_lambda": [0, 1.0],
    }),
    "LGBM_Classifier": (LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [-1, 5, 10],
        "clf__num_leaves": [31, 63, 127],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__reg_alpha": [0, 0.1],
        "clf__reg_lambda": [0, 1.0],
    }),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1) if len(y.unique()) < 10 else KFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n\U0001f539 Training {name}...")

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        is_classifier = isinstance(model, LGBMClassifier)
        scoring = "roc_auc" if is_classifier else "r2"

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False)

        try:
            y_train_use = y_train if not is_classifier else (y_train > np.median(y_train)).astype(int)
            y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)

            grid.fit(X_train, y_train_use)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            y_pred = best_model.predict(X_test)

            if is_classifier:
                y_pred_binary = (y_pred > 0.5).astype(int)
                y_proba = best_model.predict_proba(X_test)[:, 1]
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "Accuracy": accuracy_score(y_test_use, y_pred_binary),
                    "Recall": recall_score(y_test_use, y_pred_binary),
                    "Precision": precision_score(y_test_use, y_pred_binary),
                    "F1 Score": f1_score(y_test_use, y_pred_binary),
                    "ROC AUC": roc_auc_score(y_test_use, y_proba),
                    "RMSE": np.sqrt(mean_squared_error(y_test_use, y_pred)),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R\u00b2 Score": r2_score(y_test_use, y_pred),
                }
            else:
                n = len(y_test_use)
                p = X_train.shape[1]
                rss = np.sum((y_test_use - y_pred) ** 2)
                mse = mean_squared_error(y_test_use, y_pred)
                aic = n * np.log(rss / n) + 2 * p
                bic = n * np.log(rss / n) + p * np.log(n)
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "RMSE": np.sqrt(mse),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R\u00b2 Score": r2_score(y_test_use, y_pred),
                    "AIC": aic,
                    "BIC": bic
                }

            results.append(metrics)
        except Exception as e:
            print(f"\u26a0\ufe0f Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n\u2705 Model Evaluation Summary:")
results_df

In [ ]:
# Best Model by R\u00b2

regression_results = [r for r in results if 'R\u00b2 Score' in r and 'Accuracy' not in r]
if regression_results:
    best_model_name = max(regression_results, key=lambda x: x['R\u00b2 Score'])['Model']
    best_model_index = [i for i, (n, p) in enumerate(pipelines) if n == best_model_name][0]
    best_model_pipeline = pipelines[best_model_index][1]
    print(f"\n\U0001f3c6 Best Regression Model: {best_model_name}")
else:
    classification_results = [r for r in results if 'Accuracy' in r]
    best_model_name = max(classification_results, key=lambda x: x['F1 Score'])['Model']
    best_model_index = [i for i, (n, p) in enumerate(pipelines) if n == best_model_name][0]
    best_model_pipeline = pipelines[best_model_index][1]
    print(f"\n\U0001f3c6 Best Classification Model: {best_model_name}")

In [ ]:
# Feature Importance

for name, pipe in pipelines:
    print(f"\n\U0001f4ca {name} Feature Importance:")
    
    lgbm_model = pipe.named_steps["clf"]
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Split importance
    split_imp = lgbm_model.booster_.feature_importance(importance_type="split")
    df_split = pd.DataFrame({
        "feature": feature_names,
        "importance": split_imp
    }).sort_values("importance", ascending=False)
    
    sns.barplot(data=df_split, x="importance", y="feature", ax=axes[0])
    axes[0].set_title(f"{name} - Split Importance")
    
    # Gain importance
    gain_imp = lgbm_model.booster_.feature_importance(importance_type="gain")
    df_gain = pd.DataFrame({
        "feature": feature_names,
        "importance": gain_imp
    }).sort_values("importance", ascending=False)
    
    sns.barplot(data=df_gain, x="importance", y="feature", ax=axes[1])
    axes[1].set_title(f"{name} - Gain Importance")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Early Stopping Demo

print("\U0001f539 Early Stopping Demo (Regression)...")

reg_pipe = [p for n, p in pipelines if "Regressor" in n]
if reg_pipe:
    best_params = reg_pipe[0].named_steps["clf"].get_params()
    
    X_train_es, X_val, y_train_es, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )
    
    X_train_transformed = preprocessor.fit_transform(X_train_es)
    X_val_transformed = preprocessor.transform(X_val)
    
    lgbm_es = LGBMRegressor(
        n_estimators=1000,
        max_depth=best_params.get("max_depth", -1),
        num_leaves=best_params.get("num_leaves", 31),
        learning_rate=best_params.get("learning_rate", 0.05),
        subsample=best_params.get("subsample", 0.8),
        colsample_bytree=best_params.get("colsample_bytree", 0.8),
        random_state=42,
        verbosity=-1,
    )
    
    lgbm_es.fit(
        X_train_transformed, y_train_es,
        eval_set=[(X_train_transformed, y_train_es), (X_val_transformed, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    print(f"\u2705 Best iteration: {lgbm_es.best_iteration_}")
    print(f"   Best RMSE: {lgbm_es.best_score_['valid_1']['rmse']:.4f}")

In [ ]:
# Learning Curves

if reg_pipe:
    eval_results = lgbm_es.evals_result_
    
    plt.figure(figsize=(10, 5))
    plt.plot(eval_results["valid_0"]["rmse"], label="Train")
    plt.plot(eval_results["valid_1"]["rmse"], label="Validation")
    plt.axvline(lgbm_es.best_iteration_, color='r', linestyle='--', label=f"Best iteration: {lgbm_es.best_iteration_}")
    plt.xlabel("Boosting Round")
    plt.ylabel("RMSE")
    plt.title("LightGBM Learning Curves")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Regression Diagnostics

def regression_diagnostics(model, X, y, name="Model"):
    y_pred = model.predict(X)
    residuals = y - y_pred
    fitted = y_pred

    plt.figure(figsize=(14, 10))

    # Residuals vs Fitted
    plt.subplot(2, 2, 1)
    sns.scatterplot(x=fitted, y=residuals, alpha=0.4)
    plt.axhline(0, color='r', linestyle='--')
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{name} - Residuals vs Fitted")

    # Residuals Distribution
    plt.subplot(2, 2, 2)
    sns.histplot(residuals, kde=True, bins=30)
    plt.xlabel("Residuals")
    plt.title(f"{name} - Residuals Distribution")

    # Predicted vs Actual
    plt.subplot(2, 2, 3)
    sns.scatterplot(x=y, y=y_pred, alpha=0.4)
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{name} - Predicted vs Actual")

    # Scale-Location
    plt.subplot(2, 2, 4)
    sns.scatterplot(x=fitted, y=np.sqrt(np.abs(residuals)), alpha=0.4)
    plt.xlabel("Fitted values")
    plt.ylabel("Sqrt(|Residuals|)")
    plt.title(f"{name} - Scale-Location")

    plt.tight_layout()
    plt.show()


reg_pipelines = [(n, p) for n, p in pipelines if "Regressor" in n]
if reg_pipelines:
    for name, pipe in reg_pipelines:
        regression_diagnostics(pipe, X_train_full, y_train_full, f"{name} (Train)")
        regression_diagnostics(pipe, X_test, y_test, f"{name} (Test)")

In [ ]:
# Classification Diagnostics

def classification_diagnostics(model, X, y_true, name="Model"):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    y_pred_binary = (y_pred > 0.5).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred_binary)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"{name} - Confusion Matrix")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc_score = roc_auc_score(y_true, y_proba)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
    axes[1].plot([0, 1], [0, 1], 'r--')
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{name} - ROC Curve")
    axes[1].legend()

    # Precision-Recall Curve
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    axes[2].plot(rec, prec)
    axes[2].set_xlabel("Recall")
    axes[2].set_ylabel("Precision")
    axes[2].set_title(f"{name} - Precision-Recall Curve")

    plt.tight_layout()
    plt.show()

    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred_binary))


clf_pipelines = [(n, p) for n, p in pipelines if "Classifier" in n]
if clf_pipelines:
    for name, pipe in clf_pipelines:
        y_train_cls = (y_train_full > np.median(y_train_full)).astype(int)
        y_test_cls = (y_test > np.median(y_test)).astype(int)
        classification_diagnostics(pipe, X_train_full, y_train_cls, f"{name} (Train)")
        classification_diagnostics(pipe, X_test, y_test_cls, f"{name} (Test)")

In [ ]:
# Profile Plots

def plot_profiles(best_pipeline, X, y_true):
    y_pred = best_pipeline.predict(X)
    
    n_cols = 3
    n_features = X.shape[1]
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()  

    for i, col in enumerate(X.columns):
        df = pd.DataFrame({
            col: X[col],
            "y_true": y_true,
            "y_pred": y_pred
        })

        grouped = df.groupby(col).agg(
            count=("y_true", "size"),
            avg_true=("y_true", "mean"),
            avg_pred=("y_pred", "mean")
        ).reset_index().sort_values(col)

        ax1 = axes[i]
        ax1.bar(grouped[col], grouped["count"], alpha=0.3)
        ax1.set_xlabel(col)
        ax1.set_ylabel("Population")

        ax2 = ax1.twinx()
        ax2.plot(grouped[col], grouped["avg_true"], marker="o", label="Actual Target")
        ax2.plot(grouped[col], grouped["avg_pred"], marker="o", linestyle="--", label="Predicted Target")
        ax2.set_ylabel("Target Value")

        ax1.set_title(col)
        ax2.legend(loc="upper right")


    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()


plot_profiles(best_model_pipeline, X_train_full, y_train_full)

In [ ]:
# future work:
#   native categorical: pass categorical_feature param instead of one-hot encoding
#   SHAP values for detailed feature explanations
#   dart boosting type for regularization (boosting_type='dart')
#   Optuna integration for efficient hyperparameter search
#   GPU acceleration: device='gpu' for faster training